# Text (label) embeddings for the capacity experiment

Encodes a large vocabulary of candidate labels with the CLIP text encoder and saves them to one `.npz`:

1. the 1000 ImageNet class names shipped with `open_clip`;
2. all nouns of WordNet via `nltk` (about 80k lemmas; set `MAX_WORDNET` to subsample).

Everything is text-only, so it takes seconds on a GPU and a few minutes on a CPU.

Output `.npz` keys: `T_imagenet` `[1000, d]`, `names_imagenet`; `T_wordnet` `[M, d]`, `names_wordnet`; plus `model`, `pretrained`, `template`.

In [ ]:
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch", "open_clip_torch", "numpy", "nltk", "tqdm"], check=False)

In [ ]:
# ---------------- parameters ----------------
OUT         = "text_embeddings.npz"
MODEL       = "ViT-B-32"
PRETRAINED  = "laion2b_s34b_b79k"     # must match the image embeddings
TEMPLATE    = "a photo of a {}."
MAX_WORDNET = 0                        # 0 = all WordNet nouns (~80k); e.g. 20000 to subsample
BATCH       = 512

In [ ]:
import numpy as np, torch, open_clip
from tqdm.auto import tqdm

device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print("device:", device)
model, _, _ = open_clip.create_model_and_transforms(MODEL, pretrained=PRETRAINED)
model = model.to(device).eval()
tokenizer = open_clip.get_tokenizer(MODEL)

def encode(names):
    prompts = [TEMPLATE.format(n.replace("_", " ")) for n in names]
    out = []
    with torch.no_grad():
        for i in tqdm(range(0, len(prompts), BATCH), desc="text"):
            toks = tokenizer(prompts[i:i+BATCH]).to(device)
            out.append(model.encode_text(toks).float().cpu().numpy())
    return np.concatenate(out, 0).astype(np.float32)

In [ ]:
# 1. ImageNet-1k class names (built into open_clip)
from open_clip import IMAGENET_CLASSNAMES
names_imagenet = list(IMAGENET_CLASSNAMES)
T_imagenet = encode(names_imagenet)
print("ImageNet-1k:", T_imagenet.shape)

In [ ]:
# 2. WordNet nouns
import nltk
nltk.download("wordnet", quiet=True)
from nltk.corpus import wordnet as wn

nouns = sorted({l.name() for s in wn.all_synsets("n") for l in s.lemmas()
                if l.name().isascii() and l.name().replace("_", "").isalpha()})
print("WordNet noun lemmas:", len(nouns))
if MAX_WORDNET and MAX_WORDNET < len(nouns):
    rng = np.random.default_rng(0)
    nouns = sorted(rng.choice(nouns, size=MAX_WORDNET, replace=False).tolist())
    print("subsampled to", len(nouns))
T_wordnet = encode(nouns)
print("WordNet:", T_wordnet.shape)

In [ ]:
np.savez_compressed(OUT,
    T_imagenet=T_imagenet, names_imagenet=np.array(names_imagenet),
    T_wordnet=T_wordnet, names_wordnet=np.array(nouns),
    model=MODEL, pretrained=PRETRAINED, template=TEMPLATE)
print("saved", OUT, f"({(T_imagenet.nbytes + T_wordnet.nbytes)/1e6:.0f} MB of embeddings)")

# sanity check: the embedding of "dog" should be closer to "puppy" than to "airplane"
def emb(w): return T_wordnet[nouns.index(w)] / np.linalg.norm(T_wordnet[nouns.index(w)])
try:
    print("cos(dog,puppy)=%.3f  cos(dog,airplane)=%.3f" % (emb("dog")@emb("puppy"), emb("dog")@emb("airplane")))
except ValueError:
    pass